<a href="https://colab.research.google.com/github/zeugirdoR/ricci-flow-tokenization/blob/main/minimal_ricci_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
MINIMAL PROOF OF CONCEPT - Graph Ricci Flow Tokenization

Strategy: Start TINY, make it work, then scale up!

Test on 1000 chars, 50 merges
Should take ~30 seconds on CPU, ~5 seconds on A100
"""

import numpy as np
from collections import Counter
import time

print("="*80)
print("MINIMAL RICCI FLOW PROOF-OF-CONCEPT")
print("="*80)
print()

# =============================================================================
# ULTRA-SIMPLE SINKHORN (Just enough to work)
# =============================================================================

def simple_wasserstein(p, q, max_iter=20):
    """
    Ultra-simplified Wasserstein for proof-of-concept.
    Just needs to give reasonable rankings.
    """
    n = len(p)

    # L1 distance (simplest)
    # W1(p,q) ≈ Σ |p[i] - q[i]|
    return np.sum(np.abs(p - q))

# =============================================================================
# MINIMAL RICCI CURVATURE
# =============================================================================

def minimal_ricci_curvature(i, j, transition, alpha=0.5):
    """
    Minimal Ollivier-Ricci curvature.

    Just enough to capture community structure!
    """
    n = transition.shape[0]

    # Lazy random walk distributions
    mu_i = np.zeros(n)
    mu_i[i] = 1 - alpha
    mu_i += alpha * transition[i, :]

    mu_j = np.zeros(n)
    mu_j[j] = 1 - alpha
    mu_j += alpha * transition[j, :]

    # Wasserstein distance (simplified)
    W = simple_wasserstein(mu_i, mu_j)

    # Ricci curvature
    kappa = 1.0 - W

    return kappa

# =============================================================================
# MINIMAL TOKENIZER
# =============================================================================

def minimal_ricci_tokenizer(text, num_merges=50, alpha=0.5, verbose=True):
    """
    Minimal working Ricci flow tokenizer.

    Just enough to prove the concept works!
    """

    if verbose:
        print(f"Text: {len(text)} chars")
        print(f"Merges: {num_merges}")
        print(f"Alpha: {alpha}")
        print()

    tokens = list(text.encode('utf-8'))
    vocab_size = 256
    merges = []

    start = time.time()

    for merge_num in range(num_merges):
        # Build graph
        adjacency = np.zeros((vocab_size, vocab_size))
        for i in range(len(tokens) - 1):
            adjacency[tokens[i], tokens[i+1]] += 1

        # Transition matrix
        row_sums = adjacency.sum(axis=1, keepdims=True)
        transition = adjacency / (row_sums + 1e-10)

        # Find edges with frequency >= 2
        edges = []
        for i in range(vocab_size):
            for j in range(vocab_size):
                if adjacency[i, j] >= 2:
                    edges.append((i, j, adjacency[i, j]))

        if not edges:
            if verbose:
                print(f"Stopped at {merge_num} merges (no more edges)")
            break

        # Compute curvatures
        curvatures = []
        for i, j, freq in edges:
            kappa = minimal_ricci_curvature(i, j, transition, alpha)
            curvatures.append((i, j, kappa, freq))

        # Sort by curvature AND frequency (both matter!)
        curvatures.sort(key=lambda x: (x[2], x[3]), reverse=True)

        # Best edge
        best_i, best_j, best_kappa, freq = curvatures[0]

        # Decode
        try:
            token = bytes([best_i, best_j]).decode('utf-8', errors='replace')
        except:
            token = f"[{best_i},{best_j}]"

        merges.append((token, best_i, best_j, best_kappa, freq))

        # Merge
        new_token = vocab_size
        vocab_size += 1

        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best_i and tokens[i+1] == best_j:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        # Progress every 10 merges
        if verbose and (merge_num + 1) % 10 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            print(f"  {merge_num+1:3d} merges: κ={best_kappa:.4f}, "
                  f"'{token}' (freq={freq:.0f}), comp={comp:.1f}%")

    elapsed = time.time() - start
    final_comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100

    if verbose:
        print()
        print(f"✓ Done in {elapsed:.1f}s")
        print(f"  Compression: {final_comp:.1f}%")
        print()

    return merges, tokens, final_comp

# =============================================================================
# BPE BASELINE (for comparison)
# =============================================================================

def simple_bpe(text, num_merges=50):
    """Simple BPE for comparison."""
    tokens = list(text.encode('utf-8'))
    merges = []

    for _ in range(num_merges):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1

        if not pairs:
            break

        best = max(pairs, key=pairs.get)
        merges.append(best)

        new_token = 256 + len(merges) - 1
        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best[0] and tokens[i+1] == best[1]:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
    return merges, tokens, comp

# =============================================================================
# PROOF-OF-CONCEPT TEST
# =============================================================================

if __name__ == "__main__":

    # TINY corpus for proof-of-concept
    tiny_text = """Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversations?'

So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure
of making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her."""

    print("="*80)
    print("TINY CORPUS - PROOF OF CONCEPT")
    print("="*80)
    print(f"Corpus: {len(tiny_text)} characters")
    print()

    # Test different alphas
    print("="*80)
    print("TESTING DIFFERENT ALPHA VALUES")
    print("="*80)
    print()

    results = []
    for alpha in [0.3, 0.5, 0.7]:
        print(f"\n{'='*40}")
        print(f"Alpha = {alpha}")
        print('='*40)

        merges_r, tokens_r, comp_r = minimal_ricci_tokenizer(
            tiny_text,
            num_merges=50,
            alpha=alpha,
            verbose=True
        )

        # Show top 10 tokens
        print("Top 10 tokens:")
        for i, (tok, _, _, kappa, freq) in enumerate(merges_r[:10], 1):
            print(f"  {i:2d}. '{tok:>4s}' (κ={kappa:.4f}, freq={freq:>3.0f})")

        results.append((alpha, comp_r, merges_r))

    # BPE comparison
    print("\n" + "="*80)
    print("BPE BASELINE")
    print("="*80)
    merges_bpe, tokens_bpe, comp_bpe = simple_bpe(tiny_text, num_merges=50)
    print(f"BPE compression: {comp_bpe:.1f}%")
    print()
    print("Top 10 BPE tokens:")
    for i, (b1, b2) in enumerate(merges_bpe[:10], 1):
        try:
            tok = bytes([b1, b2]).decode('utf-8', errors='replace')
        except:
            tok = f"[{b1},{b2}]"
        print(f"  {i:2d}. '{tok}'")

    # Results
    print("\n" + "="*80)
    print("RESULTS")
    print("="*80)
    print()

    best_alpha, best_comp, best_merges = max(results, key=lambda x: x[1])

    print(f"{'Alpha':<10} {'Compression':<15}")
    print("-"*25)
    for alpha, comp, _ in results:
        marker = " ← BEST" if alpha == best_alpha else ""
        print(f"{alpha:<10.1f} {comp:<14.1f}%{marker}")
    print(f"{'BPE':<10} {comp_bpe:<14.1f}%")

    print()
    improvement = best_comp - comp_bpe
    if improvement > 0:
        print(f"✓✓ RICCI WINS by {improvement:.1f}%!")
        print(f"   Best alpha: {best_alpha}")
    elif improvement > -2:
        print(f"≈ CLOSE! (within {abs(improvement):.1f}%)")
        print(f"   Tuning may improve further")
    else:
        print(f"⚠ BPE ahead by {abs(improvement):.1f}%")
        print(f"   Need different approach or more tuning")

    print()
    print("="*80)
    print("CONCLUSION")
    print("="*80)
    print()

    if improvement > -2:
        print("✓ Proof-of-concept WORKS!")
        print("✓ Ricci curvature captures community structure")
        print("✓ Ready to scale up with A100")
        print()
        print("Next steps:")
        print("  1. Run on 10K corpus (10× larger)")
        print("  2. Try 200 merges")
        print("  3. Fine-tune alpha")
        print("  4. Scale to full Alice (150K)")
    else:
        print("⚠ Need to investigate further")
        print()
        print("Things to try:")
        print("  1. Different alpha values")
        print("  2. Different Wasserstein approximation")
        print("  3. Weight frequency more heavily")

    print()
    print("="*80)


MINIMAL RICCI FLOW PROOF-OF-CONCEPT

TINY CORPUS - PROOF OF CONCEPT
Corpus: 601 characters

TESTING DIFFERENT ALPHA VALUES


Alpha = 0.3
Text: 601 chars
Merges: 50
Alpha: 0.3

   10 merges: κ=-0.4000, '[262,105]' (freq=2), comp=5.3%
   20 merges: κ=-0.4888, 'e ' (freq=18), comp=13.1%
   30 merges: κ=-0.5971, '[273,115]' (freq=2), comp=23.5%
   40 merges: κ=-0.4000, '[294,116]' (freq=2), comp=27.0%
   50 merges: κ=-0.6374, 't ' (freq=8), comp=31.8%

✓ Done in 1.2s
  Compression: 31.8%

Top 10 tokens:
   1. '  ee' (κ=1.0000, freq=  3)
   2. '  oo' (κ=1.0000, freq=  2)
   3. '  tt' (κ=1.0000, freq=  2)
   4. '  er' (κ=-0.3640, freq= 13)
   5. '[258,105]' (κ=-0.3850, freq=  2)
   6. '[257,107]' (κ=-0.4000, freq=  2)
   7. '  Al' (κ=-0.4000, freq=  2)
   8. '[260,110]' (κ=-0.4000, freq=  2)
   9. '[263,103]' (κ=-0.4000, freq=  2)
  10. '[262,105]' (κ=-0.4000, freq=  2)

Alpha = 0.5
Text: 601 chars
Merges: 50
Alpha: 0.5

   10 merges: κ=-0.0000, '[262,105]' (freq=2), comp=5.3%
   20 merges: 

In [2]:
"""
IMPROVED: Hybrid Ricci-Frequency Scoring
"""
import numpy as np
from collections import Counter
import time

def simple_wasserstein(p, q):
    """L1 distance approximation."""
    return np.sum(np.abs(p - q))

def minimal_ricci_curvature(i, j, transition, alpha=0.7):
    """Minimal Ollivier-Ricci curvature."""
    n = transition.shape[0]

    mu_i = np.zeros(n)
    mu_i[i] = 1 - alpha
    mu_i += alpha * transition[i, :]

    mu_j = np.zeros(n)
    mu_j[j] = 1 - alpha
    mu_j += alpha * transition[j, :]

    W = simple_wasserstein(mu_i, mu_j)
    kappa = 1.0 - W

    return kappa

def hybrid_ricci_tokenizer(text, num_merges=50, alpha=0.7, beta=0.5, verbose=True):
    """
    Hybrid: Score = β·κ + (1-β)·log(freq)

    beta=0: Pure frequency (like BPE)
    beta=0.5: Balanced (geometry + frequency)
    beta=1: Pure curvature (what we tried before)
    """

    if verbose:
        print(f"Text: {len(text)} chars")
        print(f"Merges: {num_merges}")
        print(f"Alpha: {alpha}, Beta: {beta}")
        print()

    tokens = list(text.encode('utf-8'))
    vocab_size = 256
    merges = []

    start = time.time()

    for merge_num in range(num_merges):
        # Build graph
        adjacency = np.zeros((vocab_size, vocab_size))
        for i in range(len(tokens) - 1):
            adjacency[tokens[i], tokens[i+1]] += 1

        # Transition matrix
        row_sums = adjacency.sum(axis=1, keepdims=True)
        transition = adjacency / (row_sums + 1e-10)

        # Find edges
        edges = []
        for i in range(vocab_size):
            for j in range(vocab_size):
                if adjacency[i, j] >= 2:
                    edges.append((i, j, adjacency[i, j]))

        if not edges:
            break

        # Compute HYBRID scores
        max_freq = max(e[2] for e in edges)
        scores = []
        for i, j, freq in edges:
            kappa = minimal_ricci_curvature(i, j, transition, alpha)

            # Normalize curvature to [0, 1]
            kappa_norm = (kappa + 1) / 2

            # Normalize frequency to [0, 1]
            freq_norm = freq / max_freq

            # Hybrid score
            hybrid = beta * kappa_norm + (1 - beta) * freq_norm

            scores.append((i, j, hybrid, kappa, freq))

        # Sort by hybrid score
        scores.sort(key=lambda x: x[2], reverse=True)

        # Best edge
        best_i, best_j, hybrid, kappa, freq = scores[0]

        # Decode
        try:
            token = bytes([best_i, best_j]).decode('utf-8', errors='replace')
        except:
            token = f"[{best_i},{best_j}]"

        merges.append((token, best_i, best_j, hybrid, kappa, freq))

        # Merge
        new_token = vocab_size
        vocab_size += 1

        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best_i and tokens[i+1] == best_j:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 10 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            print(f"  {merge_num+1:3d}: score={hybrid:.4f}, κ={kappa:.4f}, "
                  f"'{token}' (freq={freq:.0f}), comp={comp:.1f}%")

    elapsed = time.time() - start
    final_comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100

    if verbose:
        print(f"\n✓ Done in {elapsed:.1f}s, Compression: {final_comp:.1f}%\n")

    return merges, tokens, final_comp

def simple_bpe(text, num_merges=50):
    """BPE baseline."""
    tokens = list(text.encode('utf-8'))
    merges = []

    for _ in range(num_merges):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)
        new_token = 256 + len(merges) - 1
        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best[0] and tokens[i+1] == best[1]:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

    comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
    return merges, tokens, comp

# TEST
tiny_text = """Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversations?'

So she was considering in her own mind (as well as she could, for the
hot day made her feel very sleepy and stupid), whether the pleasure
of making a daisy-chain would be worth the trouble of getting up and
picking the daisies, when suddenly a White Rabbit with pink eyes ran
close by her."""

print("="*80)
print("HYBRID RICCI-FREQUENCY SCORING")
print("="*80)
print()

# Test different beta values
results = []
for beta in [0.0, 0.3, 0.5, 0.7, 1.0]:
    print(f"Beta = {beta} (β·geometry + (1-β)·frequency)")
    print("-"*40)
    merges, tokens, comp = hybrid_ricci_tokenizer(tiny_text, num_merges=50, alpha=0.7, beta=beta, verbose=True)

    print("Top 5 tokens:")
    for i, (tok, _, _, score, kappa, freq) in enumerate(merges[:5], 1):
        print(f"  {i}. '{tok}' (score={score:.3f}, κ={kappa:.3f}, freq={freq:.0f})")
    print()

    results.append((beta, comp))

# BPE
print("="*80)
print("BPE BASELINE")
print("="*80)
_, _, comp_bpe = simple_bpe(tiny_text, 50)
print(f"BPE: {comp_bpe:.1f}%\n")

# Results
print("="*80)
print("RESULTS")
print("="*80)
print()
print(f"{'Beta':<10} {'Compression':<15} {'vs BPE':<15}")
print("-"*40)
for beta, comp in results:
    diff = comp - comp_bpe
    marker = " ✓" if diff >= 0 else ""
    print(f"{beta:<10.1f} {comp:<14.1f}% {diff:>+6.1f}%{marker}")
print(f"{'BPE':<10} {comp_bpe:<14.1f}%")

best_beta, best_comp = max(results, key=lambda x: x[1])
print()
if best_comp >= comp_bpe:
    print(f"🎉 HYBRID WINS! Best beta={best_beta}")
elif best_comp >= comp_bpe - 2:
    print(f"✓ CLOSE! Best beta={best_beta} (within 2%)")
else:
    print(f"⚠ BPE ahead, but beta={best_beta} is best")

HYBRID RICCI-FREQUENCY SCORING

Beta = 0.0 (β·geometry + (1-β)·frequency)
----------------------------------------
Text: 601 chars
Merges: 50
Alpha: 0.7, Beta: 0.0

   10: score=1.0000, κ=-0.2734, 'on' (freq=7), comp=19.1%
   20: score=1.0000, κ=-0.6138, 'o ' (freq=5), comp=29.0%
   30: score=1.0000, κ=-0.4559, 'a ' (freq=3), comp=36.1%
   40: score=1.0000, κ=-0.3969, '[119,272]' (freq=3), comp=41.1%
   50: score=1.0000, κ=-0.2600, 'ge' (freq=2), comp=44.9%

✓ Done in 1.6s, Compression: 44.9%

Top 5 tokens:
  1. 'e ' (score=1.000, κ=-0.061, freq=21)
  2. 'in' (score=1.000, κ=-0.060, freq=16)
  3. 'er' (score=1.000, κ=0.008, freq=13)
  4. 'th' (score=1.000, κ=-0.109, freq=13)
  5. 'd ' (score=1.000, κ=-0.113, freq=10)

Beta = 0.3 (β·geometry + (1-β)·frequency)
----------------------------------------
Text: 601 chars
Merges: 50
Alpha: 0.7, Beta: 0.3

   10: score=0.8200, κ=-0.2000, '[259,256]' (freq=7), comp=19.1%
   20: score=0.7900, κ=-0.4000, '[273,32]' (freq=5), comp=29.0%
   30: sco

In [3]:
"""
SCALING TEST - 10K corpus where geometry can help!
"""
import numpy as np
from collections import Counter
import time

def simple_wasserstein(p, q):
    return np.sum(np.abs(p - q))

def minimal_ricci_curvature(i, j, transition, alpha=0.7):
    n = transition.shape[0]
    mu_i = np.zeros(n)
    mu_i[i] = 1 - alpha
    mu_i += alpha * transition[i, :]
    mu_j = np.zeros(n)
    mu_j[j] = 1 - alpha
    mu_j += alpha * transition[j, :]
    W = simple_wasserstein(mu_i, mu_j)
    return 1.0 - W

def hybrid_ricci_tokenizer(text, num_merges=200, alpha=0.7, beta=0.3, verbose=True):
    """Hybrid with progress tracking."""

    if verbose:
        print(f"Corpus: {len(text)} chars, Merges: {num_merges}, α={alpha}, β={beta}")
        print()

    tokens = list(text.encode('utf-8'))
    vocab_size = 256
    merges = []
    start = time.time()

    for merge_num in range(num_merges):
        adjacency = np.zeros((vocab_size, vocab_size))
        for i in range(len(tokens) - 1):
            adjacency[tokens[i], tokens[i+1]] += 1

        row_sums = adjacency.sum(axis=1, keepdims=True)
        transition = adjacency / (row_sums + 1e-10)

        edges = []
        for i in range(vocab_size):
            for j in range(vocab_size):
                if adjacency[i, j] >= 2:
                    edges.append((i, j, adjacency[i, j]))

        if not edges:
            if verbose:
                print(f"Stopped at {merge_num} merges")
            break

        max_freq = max(e[2] for e in edges)
        scores = []
        for i, j, freq in edges:
            kappa = minimal_ricci_curvature(i, j, transition, alpha)
            kappa_norm = (kappa + 1) / 2
            freq_norm = freq / max_freq
            hybrid = beta * kappa_norm + (1 - beta) * freq_norm
            scores.append((i, j, hybrid, kappa, freq))

        scores.sort(key=lambda x: x[2], reverse=True)
        best_i, best_j, hybrid, kappa, freq = scores[0]

        try:
            token = bytes([best_i, best_j]).decode('utf-8', errors='replace')
        except:
            token = f"[{best_i},{best_j}]"

        merges.append((token, best_i, best_j, hybrid, kappa, freq))

        new_token = vocab_size
        vocab_size += 1

        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best_i and tokens[i+1] == best_j:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 50 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            elapsed = time.time() - start
            speed = (merge_num + 1) / elapsed
            print(f"  {merge_num+1:3d} merges: '{token:>4s}' comp={comp:.1f}%, {speed:.1f} m/s")

    elapsed = time.time() - start
    final_comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100

    if verbose:
        print(f"\n✓ Done in {elapsed:.1f}s, Compression: {final_comp:.1f}%\n")

    return merges, tokens, final_comp

def simple_bpe(text, num_merges=200, verbose=True):
    """BPE with progress."""
    if verbose:
        print(f"BPE: {len(text)} chars, {num_merges} merges")
        print()

    tokens = list(text.encode('utf-8'))
    merges = []
    start = time.time()

    for merge_num in range(num_merges):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)

        new_token = 256 + len(merges) - 1
        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best[0] and tokens[i+1] == best[1]:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 50 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            elapsed = time.time() - start
            speed = (merge_num + 1) / elapsed
            try:
                tok = bytes(best).decode('utf-8', errors='replace')
            except:
                tok = f"[{best[0]},{best[1]}]"
            print(f"  {merge_num+1:3d} merges: '{tok:>4s}' comp={comp:.1f}%, {speed:.1f} m/s")

    elapsed = time.time() - start
    comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
    if verbose:
        print(f"\n✓ Done in {elapsed:.1f}s, Compression: {comp:.1f}%\n")
    return merges, tokens, comp

# Download or use your Alice text
print("="*80)
print("10K CORPUS TEST - Where Geometry Can Help!")
print("="*80)
print()

# If you have alice downloaded:
try:
    import urllib.request
    url = "https://www.gutenberg.org/files/11/11-0.txt"
    with urllib.request.urlopen(url) as response:
        alice_full = response.read().decode('utf-8')
    start_idx = alice_full.find("CHAPTER I")
    alice = alice_full[start_idx:start_idx+10000]
    print("✓ Using Alice in Wonderland")
except:
    # Or use repeated text
    alice = ("""Alice was beginning to get very tired of sitting by her sister on the
bank, and of having nothing to do: once or twice she had peeped into the
book her sister was reading, but it had no pictures or conversations in
it, 'and what is the use of a book,' thought Alice 'without pictures or
conversations?'""" * 20)
    print("✓ Using sample text")

print(f"Corpus: {len(alice)} characters\n")

# Test beta values
print("="*80)
print("TESTING HYBRID ON LARGER CORPUS")
print("="*80)
print()

results = []
for beta in [0.0, 0.2, 0.4]:
    print(f"\n{'='*40}")
    print(f"Beta = {beta}")
    print('='*40)
    _, _, comp = hybrid_ricci_tokenizer(alice, num_merges=200, alpha=0.7, beta=beta, verbose=True)
    results.append((beta, comp))

# BPE
print("="*80)
print("BPE BASELINE")
print("="*80)
_, _, comp_bpe = simple_bpe(alice, num_merges=200, verbose=True)

# Results
print("="*80)
print("FINAL RESULTS - 10K CORPUS")
print("="*80)
print()
print(f"{'Beta':<10} {'Compression':<15} {'vs BPE':<15}")
print("-"*40)
for beta, comp in results:
    diff = comp - comp_bpe
    marker = " 🎉" if diff > 0 else " ✓" if diff == 0 else ""
    print(f"{beta:<10.1f} {comp:<14.1f}% {diff:>+6.1f}%{marker}")
print(f"{'BPE':<10} {comp_bpe:<14.1f}%")

best_beta, best_comp = max(results, key=lambda x: x[1])
print()
if best_comp > comp_bpe:
    print(f"🎉🎉 GEOMETRY WINS! Beta={best_beta} beats BPE by {best_comp-comp_bpe:.1f}%!")
elif best_comp == comp_bpe:
    print(f"✓ Tied with BPE (corpus still small or all optimal)")
else:
    print(f"Close! Best={best_beta}, Gap={comp_bpe-best_comp:.1f}%")


10K CORPUS TEST - Where Geometry Can Help!

✓ Using Alice in Wonderland
Corpus: 10000 characters

TESTING HYBRID ON LARGER CORPUS


Beta = 0.0
Corpus: 10000 chars, Merges: 200, α=0.7, β=0.0

   50 merges: '  

' comp=34.0%, 30.2 m/s
  100 merges: '  t
' comp=43.2%, 26.4 m/s
  150 merges: '[268,326]' comp=48.9%, 23.2 m/s
  200 merges: '  tt' comp=52.9%, 19.2 m/s

✓ Done in 10.4s, Compression: 52.9%


Beta = 0.2
Corpus: 10000 chars, Merges: 200, α=0.7, β=0.2

   50 merges: '  e
' comp=33.9%, 24.3 m/s
  100 merges: '[265,298]' comp=43.2%, 23.8 m/s
  150 merges: '  if' comp=49.1%, 22.4 m/s
  200 merges: '[100,276]' comp=53.0%, 20.8 m/s

✓ Done in 9.6s, Compression: 53.0%


Beta = 0.4
Corpus: 10000 chars, Merges: 200, α=0.7, β=0.4

   50 merges: '   m' comp=33.7%, 25.1 m/s
  100 merges: '  le' comp=42.9%, 19.3 m/s
  150 merges: '[108,381]' comp=48.7%, 19.4 m/s
  200 merges: '[32,285]' comp=52.8%, 18.9 m/s

✓ Done in 10.6s, Compression: 52.8%

BPE BASELINE
BPE: 10000 chars, 200 merges

   50

In [4]:
import numpy as np
from collections import Counter
import time

def simple_wasserstein(p, q):
    return np.sum(np.abs(p - q))

def minimal_ricci_curvature(i, j, transition, alpha=0.7):
    n = transition.shape[0]
    mu_i = np.zeros(n)
    mu_i[i] = 1 - alpha
    mu_i += alpha * transition[i, :]
    mu_j = np.zeros(n)
    mu_j[j] = 1 - alpha
    mu_j += alpha * transition[j, :]
    W = simple_wasserstein(mu_i, mu_j)
    return 1.0 - W

def hybrid_ricci_tokenizer(text, num_merges=500, alpha=0.7, beta=0.2, verbose=True):
    if verbose:
        print(f"Corpus: {len(text)} chars, Merges: {num_merges}, α={alpha}, β={beta}")
        print()

    tokens = list(text.encode('utf-8'))
    vocab_size = 256
    merges = []
    start = time.time()

    for merge_num in range(num_merges):
        adjacency = np.zeros((vocab_size, vocab_size))
        for i in range(len(tokens) - 1):
            adjacency[tokens[i], tokens[i+1]] += 1

        row_sums = adjacency.sum(axis=1, keepdims=True)
        transition = adjacency / (row_sums + 1e-10)

        edges = []
        for i in range(vocab_size):
            for j in range(vocab_size):
                if adjacency[i, j] >= 2:
                    edges.append((i, j, adjacency[i, j]))

        if not edges:
            if verbose:
                print(f"Stopped at {merge_num} merges")
            break

        max_freq = max(e[2] for e in edges)
        scores = []
        for i, j, freq in edges:
            kappa = minimal_ricci_curvature(i, j, transition, alpha)
            kappa_norm = (kappa + 1) / 2
            freq_norm = freq / max_freq
            hybrid = beta * kappa_norm + (1 - beta) * freq_norm
            scores.append((i, j, hybrid, kappa, freq))

        scores.sort(key=lambda x: x[2], reverse=True)
        best_i, best_j, hybrid, kappa, freq = scores[0]

        try:
            token = bytes([best_i, best_j]).decode('utf-8', errors='replace')
        except:
            token = f"[{best_i},{best_j}]"

        merges.append((token, best_i, best_j, hybrid, kappa, freq))

        new_token = vocab_size
        vocab_size += 1

        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best_i and tokens[i+1] == best_j:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 100 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            elapsed = time.time() - start
            speed = (merge_num + 1) / elapsed
            print(f"  {merge_num+1:3d} merges: comp={comp:.1f}%, {speed:.1f} m/s")

    elapsed = time.time() - start
    final_comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100

    if verbose:
        print(f"\n✓ Done in {elapsed:.1f}s, Compression: {final_comp:.2f}%")

    return merges, tokens, final_comp

def simple_bpe(text, num_merges=500, verbose=True):
    if verbose:
        print(f"BPE: {len(text)} chars, {num_merges} merges")
        print()

    tokens = list(text.encode('utf-8'))
    merges = []
    start = time.time()

    for merge_num in range(num_merges):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)

        new_token = 256 + len(merges) - 1
        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best[0] and tokens[i+1] == best[1]:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 100 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            elapsed = time.time() - start
            speed = (merge_num + 1) / elapsed
            print(f"  {merge_num+1:3d} merges: comp={comp:.1f}%, {speed:.1f} m/s")

    elapsed = time.time() - start
    comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
    if verbose:
        print(f"\n✓ Done in {elapsed:.1f}s, Compression: {comp:.2f}%")
    return merges, tokens, comp

print("="*80)
print("50K CORPUS - WHERE GEOMETRY SHOULD SHINE!")
print("="*80)
print()

import urllib.request
url = "https://www.gutenberg.org/files/11/11-0.txt"
with urllib.request.urlopen(url) as response:
    alice_full = response.read().decode('utf-8')
start_idx = alice_full.find("CHAPTER I")
alice = alice_full[start_idx:start_idx+50000]

print(f"Corpus: {len(alice)} characters")
print()

print("="*80)
print("HYBRID RICCI (Beta=0.2)")
print("="*80)
_, _, comp_ricci = hybrid_ricci_tokenizer(alice, num_merges=500, alpha=0.7, beta=0.2)

print()
print("="*80)
print("BPE BASELINE")
print("="*80)
_, _, comp_bpe = simple_bpe(alice, num_merges=500)

print()
print("="*80)
print("RESULTS - 50K CORPUS")
print("="*80)
print()
improvement = comp_ricci - comp_bpe
print(f"Hybrid Ricci: {comp_ricci:.2f}%")
print(f"BPE:          {comp_bpe:.2f}%")
print(f"Improvement:  {improvement:+.2f}%")
print()

if improvement > 0.5:
    print(f"🎉🎉 GEOMETRY WINS by {improvement:.2f}%!")
elif improvement > 0:
    print(f"✓ Geometry helps! +{improvement:.2f}%")
elif improvement > -0.5:
    print(f"≈ Essentially tied ({abs(improvement):.2f}% diff)")
else:
    print(f"Need more tuning (BPE ahead {abs(improvement):.2f}%)")

50K CORPUS - WHERE GEOMETRY SHOULD SHINE!

Corpus: 50000 characters

HYBRID RICCI (Beta=0.2)
Corpus: 50000 chars, Merges: 500, α=0.7, β=0.2

  100 merges: comp=42.9%, 11.1 m/s
  200 merges: comp=51.6%, 9.4 m/s
  300 merges: comp=56.6%, 8.3 m/s
  400 merges: comp=59.9%, 7.4 m/s
  500 merges: comp=62.4%, 6.6 m/s

✓ Done in 75.8s, Compression: 62.44%

BPE BASELINE
BPE: 50000 chars, 500 merges

  100 merges: comp=42.9%, 49.8 m/s
  200 merges: comp=51.7%, 54.3 m/s
  300 merges: comp=56.6%, 57.3 m/s
  400 merges: comp=59.9%, 59.8 m/s
  500 merges: comp=62.5%, 55.7 m/s

✓ Done in 9.0s, Compression: 62.48%

RESULTS - 50K CORPUS

Hybrid Ricci: 62.44%
BPE:          62.48%
Improvement:  -0.04%

≈ Essentially tied (0.04% diff)


In [ ]:
import numpy as np
from collections import Counter
import time

def simple_wasserstein(p, q):
    return np.sum(np.abs(p - q))

def minimal_ricci_curvature(i, j, transition, alpha=0.8):
    n = transition.shape[0]
    mu_i = np.zeros(n)
    mu_i[i] = 1 - alpha
    mu_i += alpha * transition[i, :]
    mu_j = np.zeros(n)
    mu_j[j] = 1 - alpha
    mu_j += alpha * transition[j, :]
    W = simple_wasserstein(mu_i, mu_j)
    return 1.0 - W

def hybrid_ricci_tokenizer(text, num_merges=5000, alpha=0.8, beta=0.2, verbose=True):
    if verbose:
        print(f"Corpus: {len(text)} chars, Merges: {num_merges}, α={alpha}, β={beta}")
        print()

    tokens = list(text.encode('utf-8'))
    vocab_size = 256
    merges = []
    start = time.time()

    for merge_num in range(num_merges):
        adjacency = np.zeros((vocab_size, vocab_size))
        for i in range(len(tokens) - 1):
            adjacency[tokens[i], tokens[i+1]] += 1

        row_sums = adjacency.sum(axis=1, keepdims=True)
        transition = adjacency / (row_sums + 1e-10)

        edges = []
        for i in range(vocab_size):
            for j in range(vocab_size):
                if adjacency[i, j] >= 2:
                    edges.append((i, j, adjacency[i, j]))

        if not edges:
            if verbose:
                print(f"Stopped at {merge_num} merges")
            break

        max_freq = max(e[2] for e in edges)
        scores = []
        for i, j, freq in edges:
            kappa = minimal_ricci_curvature(i, j, transition, alpha)
            kappa_norm = (kappa + 1) / 2
            freq_norm = freq / max_freq
            hybrid = beta * kappa_norm + (1 - beta) * freq_norm
            scores.append((i, j, hybrid, kappa, freq))

        scores.sort(key=lambda x: x[2], reverse=True)
        best_i, best_j, hybrid, kappa, freq = scores[0]

        try:
            token = bytes([best_i, best_j]).decode('utf-8', errors='replace')
        except:
            token = f"[{best_i},{best_j}]"

        merges.append((token, best_i, best_j, hybrid, kappa, freq))

        new_token = vocab_size
        vocab_size += 1

        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best_i and tokens[i+1] == best_j:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 1000 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            elapsed = time.time() - start
            speed = (merge_num + 1) / elapsed
            print(f"  {merge_num+1:4d} merges: comp={comp:.2f}%, {speed:.1f} m/s")

    elapsed = time.time() - start
    final_comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100

    if verbose:
        print(f"\n✓ Done in {elapsed:.0f}s, Compression: {final_comp:.2f}%")

    return merges, tokens, final_comp

def simple_bpe(text, num_merges=5000, verbose=True):
    if verbose:
        print(f"BPE: {len(text)} chars, {num_merges} merges")
        print()

    tokens = list(text.encode('utf-8'))
    merges = []
    start = time.time()

    for merge_num in range(num_merges):
        pairs = Counter()
        for i in range(len(tokens) - 1):
            pairs[(tokens[i], tokens[i+1])] += 1
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        merges.append(best)

        new_token = 256 + len(merges) - 1
        i, new_tokens = 0, []
        while i < len(tokens):
            if i < len(tokens)-1 and tokens[i] == best[0] and tokens[i+1] == best[1]:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens

        if verbose and (merge_num + 1) % 1000 == 0:
            comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
            elapsed = time.time() - start
            speed = (merge_num + 1) / elapsed
            print(f"  {merge_num+1:4d} merges: comp={comp:.2f}%, {speed:.1f} m/s")

    elapsed = time.time() - start
    comp = (1 - len(tokens) / len(text.encode('utf-8'))) * 100
    if verbose:
        print(f"\n✓ Done in {elapsed:.0f}s, Compression: {comp:.2f}%")
    return merges, tokens, comp

print("="*80)
print("FULL ALICE - 150K CORPUS")
print("="*80)
print()

import urllib.request
url = "https://www.gutenberg.org/files/11/11-0.txt"
with urllib.request.urlopen(url) as response:
    alice_full = response.read().decode('utf-8')
start_idx = alice_full.find("CHAPTER I")
alice = alice_full[start_idx:start_idx+150000]

print(f"Corpus: {len(alice)} characters")
print()

print("="*80)
print("HYBRID RICCI (α=0.8, β=0.2)")
print("="*80)
_, _, comp_ricci = hybrid_ricci_tokenizer(alice, num_merges=5000, alpha=0.8, beta=0.2)

print()
print("="*80)
print("BPE BASELINE")
print("="*80)
_, _, comp_bpe = simple_bpe(alice, num_merges=5000)

print()
print("="*80)
print("FINAL RESULTS - 150K CORPUS vs Yesterday's 77.56%!")
print("="*80)
print()
print(f"Hybrid Ricci: {comp_ricci:.2f}%")
print(f"BPE:          {comp_bpe:.2f}%")
print(f"Yesterday:    77.56%")
print()
improvement = comp_ricci - comp_bpe
print(f"Ricci vs BPE: {improvement:+.2f}%")
print()

if comp_ricci > 77.56:
    print(f"🎉🎉 BEATS YESTERDAY'S RECORD!")
elif comp_ricci > 77.0:
    print(f"✓ Approaching yesterday's performance!")
elif comp_ricci > 75.0:
    print(f"✓ Competitive with tiktoken (75.76%)!")

FULL ALICE - 150K CORPUS

Corpus: 144529 characters

HYBRID RICCI (α=0.8, β=0.2)
Corpus: 144529 chars, Merges: 5000, α=0.8, β=0.2

  1000 merges: comp=69.28%, 2.8 m/s
  2000 merges: comp=75.07%, 1.5 m/s
  3000 merges: comp=78.13%, 0.9 m/s
  4000 merges: comp=80.18%, 0.6 m/s
